# Section 2: AI Pipeline Management Demo
**ACME IM | Distribution Insights**

Demonstrates Snowpipe V2, Dynamic Tables, and Cortex AI propensity scoring.

In [ ]:
import os, tomllib, json, urllib.request, urllib.error
from cryptography.hazmat.primitives.serialization import load_pem_private_key
from snowflake.snowpark import Session
from snowflake.cortex import Complete
import snowflake.connector
import pandas as pd

# Load keypair auth from connections.toml (works locally; get_active_session() used in SiS)
_CONN_NAME = os.environ.get('SNOWFLAKE_DEFAULT_CONNECTION_NAME', 'your_connection')
with open(os.path.expanduser('~/.snowflake/connections.toml'), 'rb') as f:
    _cfg = tomllib.load(f)[_CONN_NAME]

_key_path = os.path.expanduser(_cfg['private_key_path'])
with open(_key_path, 'rb') as f:
    _private_key = load_pem_private_key(f.read(), password=None)

_params = {k: v for k, v in _cfg.items() if k != 'private_key_path'}
_params['private_key'] = _private_key

# Snowpark Session
session = Session.builder.configs(_params).create()
session.sql('USE WAREHOUSE INGEST').collect()
session.sql('USE DATABASE ANALYTICS_DEV_DB').collect()
session.sql('USE SCHEMA ANALYTICS_DEV_DB.DISTRIBUTION').collect()

# REST token for Cortex Analyst
_SF_HOST = f"{_cfg['account']}.snowflakecomputing.com"
_raw_conn = snowflake.connector.connect(**_params)
_SF_TOKEN = _raw_conn.rest.token

print(f'Connected: {session.get_current_account()}')
print(f'Context: {session.get_current_database()}.{session.get_current_schema()}')


In [ ]:
# Demo 1: Snowpipe V2 throughput and freshness (last 24 hours)
freshness_df = session.sql("""
    SELECT
        MAX(row_timestamp)                                          AS latest_event,
        DATEDIFF('second', MAX(row_timestamp), CURRENT_TIMESTAMP()) AS seconds_lag,
        COUNT(*)                                                    AS total_events_24h,
        COUNT(DISTINCT advisor_id)                                  AS unique_advisors
    FROM ANALYTICS_DEV_DB.STAGING.ADVISOR_EVENTS_RAW
    WHERE row_timestamp >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
""").to_pandas()
print('SNOWPIPE V2 THROUGHPUT (last 24 hours):')
print(freshness_df.to_string(index=False))


In [ ]:
# Demo 2: Dynamic Table health via direct row count
# (INFORMATION_SCHEMA.DYNAMIC_TABLES not available in all accounts)
dts = [
    ('ADVISOR_ENGAGEMENT_SCORE', '1 hour'),
    ('FUND_FLOW_ATTRIBUTION',    '1 day'),
    ('TERRITORY_HEAT_MAP',       '4 hours'),
]
rows = []
for tbl, lag in dts:
    try:
        cnt = session.sql(
            f'SELECT COUNT(*) AS c FROM ANALYTICS_DEV_DB.DISTRIBUTION.{tbl}'
        ).collect()[0]['C']
        rows.append({'TABLE': tbl, 'TARGET_LAG': lag, 'ROW_COUNT': cnt, 'STATUS': 'OK'})
    except Exception as e:
        rows.append({'TABLE': tbl, 'TARGET_LAG': lag, 'ROW_COUNT': 0, 'STATUS': str(e)[:60]})

dt_status = pd.DataFrame(rows)
print('DYNAMIC TABLE HEALTH:')
print(dt_status.to_string(index=False))


In [ ]:
# Demo 3: Cortex AI advisor attrition risk scoring (propensity model)
scores_df = session.sql("""
    SELECT advisor_id, advisor_name, advisor_tier,
           engagement_score, call_count_30d, meeting_count_30d,
           days_since_last_activity, aum_amount
    FROM ANALYTICS_DEV_DB.DISTRIBUTION.ADVISOR_ENGAGEMENT_SCORE
    WHERE score_date = (SELECT MAX(score_date)
                        FROM ANALYTICS_DEV_DB.DISTRIBUTION.ADVISOR_ENGAGEMENT_SCORE)
    ORDER BY aum_amount DESC NULLS LAST LIMIT 20
""").to_pandas()


def classify_risk(row):
    prompt = (
        f'Classify attrition risk: HIGH, MEDIUM, or LOW. '
        f'Engagement: {row["ENGAGEMENT_SCORE"]:.0f}/100, '
        f'Calls 30d: {row["CALL_COUNT_30D"]}, '
        f'Meetings 30d: {row["MEETING_COUNT_30D"]}, '
        f'Days since activity: {row["DAYS_SINCE_LAST_ACTIVITY"]}. '
        'Respond with one word only.'
    )
    try:
        return Complete('mistral-7b', prompt).strip().upper()[:6]
    except:
        return 'UNKNWN'


scores_df['ATTRITION_RISK'] = scores_df.head(5).apply(classify_risk, axis=1)
print('ADVISOR PROPENSITY SCORES:')
cols = ['ADVISOR_NAME', 'ADVISOR_TIER', 'ENGAGEMENT_SCORE', 'ATTRITION_RISK']
print(scores_df[cols].to_string(index=False))
